In [1]:
import pandas as pd 
import csv 
import numpy as np 
import math 
import random as rd 
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns
from datetime import datetime
from matplotlib.backends.backend_pdf import PdfPages


#lire le fichier .
def Lire_fichier(file_name):
    try:
        df = pd.read_csv(file_name)
        return df
    except Exception as e:
        print(f"Erreur lors de la traitement du fichier : {e}")
        return None
#lire la shema du df 
def Lire_fichier_shema(df):
    try:
        return df.columns
    except Exception as e:
        print(f"Erreur lors de la traitement du fichier : {e}")
        return None
#formater les text du df
def Formatter_text(df):
    df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)
    return df
#convertir les valeurs numéric du df
def Convert_Numeric_value(df,row):
    try:
        df[row] = pd.to_numeric(df[row], errors="coerce")
        return df
    except Exception as e:
        print(f"Erreur lors de la traitement du fichier : {e}")
        return None

#detect les doublons 
def Detecter_doublons(df,cle):
    try:
        duplicates_df = df[df.duplicated(subset=cle, keep=False)]
        return duplicates_df
    except Exception as e:
        print(f"Erreur lors de la traitement du fichier : {e}")
        return None

#detect les valeurs manquants 
def Detecter_les_valeurs_manquants(df):
    try:
        total_missing = df.isna().sum().sum()
        missing_rows = df[df.isna().any(axis=1)].fillna("Missing")
        return total_missing,missing_rows
    except Exception as e:
        print(f"Erreur lors de la traitement du fichier : {e}")
        return None

#detecter les valeurs negatifs du quantitées et de prix unitaire

def Detecter_les_valeurs_négatifs(df,cle):
    try:
        total_negatives = len(df[df[cle] < 0][cle])
        negatifs_rows = df[df[cle] < 0]
        return total_negatives,negatifs_rows
    except Exception as e:
        print(f"Erreur lors de la traitement du fichier : {e}")
        return None



#detecter les clés etrangeres absents
def Detecter_les_etrangers_manquants(df,cle):
    try:
        total_missing = len(df[df[cle].isna()][cle])
        missing_rows = df[df[cle].isna()].fillna('Missing')
        return total_missing,missing_rows
    except Exception as e:
        print(f"Erreur lors de la traitement du fichier : {e}")
        return None

#Nettoyage de fichier ventes.csv
def Nettoyer_Ventes(ventes_data,produits_data,magasins_data, export_dir):
        #eliminer les espaces
        Formatter_text(ventes_data)

        #Convertir les valeur numériques
        Convert_Numeric_value(ventes_data,"quantite")
        Convert_Numeric_value(ventes_data,"prix_unitaire")
        
        #nettoyage de vente_id 
        
        #si id est absent
        mask = ventes_data["vente_id"].isna()
        ventes_data.loc[mask, "status"] = "Rejeté"
        ventes_data.loc[mask, "justification"] = "Vente Id est absente !"
        ventes_data.loc[mask, "justification"] = (
            ventes_data.loc[mask, "justification"].fillna("")
            + " ; Vente Id est absente  !"
        )
        
        
        #si id est dupliqué changer id en id(1),... et le rejeté
        duplicate_mask = ventes_data["vente_id"].notna() & ventes_data["vente_id"].duplicated(keep="first")
        duplicate_number = ventes_data.groupby("vente_id").cumcount()
        
        ventes_data.loc[duplicate_mask, "vente_id"] = (
            ventes_data.loc[duplicate_mask, "vente_id"].astype("string")
            + "("
            + duplicate_number.loc[duplicate_mask].astype("string")
            + ")"
        )
        ventes_data.loc[duplicate_mask, "status"] = "Rejeté"
        ventes_data.loc[duplicate_mask, "justification"] = (
            ventes_data.loc[duplicate_mask, "justification"].fillna("")
            + " ; Vente Id est dupliqué  !"
        )
        #si date est absent
        dates = ventes_data["date"].astype("string").str.strip()
        parsed_dates = pd.to_datetime(dates, format="%Y-%m-%d", errors="coerce")
        
        missing_mask = dates.isna() | dates.eq("")
        invalid_mask = ~missing_mask & parsed_dates.isna()
        future_mask = parsed_dates.notna() & (
            parsed_dates > pd.Timestamp.today().normalize()
        )
        
        def reject_rows(mask, message):
            ventes_data.loc[mask, "status"] = "Rejeté"
        
            existing = (
                ventes_data.loc[mask, "justification"]
                .fillna("")
                .astype("string")
                .str.strip()
            )
            updated = existing.where(existing.eq(""), existing + " ; ") + message
            ventes_data.loc[mask, "justification"] = updated
        
        reject_rows(missing_mask, "Date absente")
        reject_rows(invalid_mask, "Format de date incompatible")
        reject_rows(future_mask, "Date dans le futur")
        
        ventes_data["date"] = parsed_dates
        
        
        
        #nettoyage de produit Id 
        
        #si produit_id est absent
        mask = ventes_data["produit_id"].isna()
        ventes_data.loc[mask, "status"] = "Rejeté"
        ventes_data.loc[mask, "justification"] = (
            ventes_data.loc[mask, "justification"].fillna("")
            + " ; Produit Id est absent !"
        )
        
        #si produit_id n'exist pas dans lable des produits
        mask = (ventes_data["produit_id"].notna() & ~ventes_data["produit_id"].isin(produits_data["produit_id"]))
        ventes_data.loc[mask, "status"] = "Rejeté"
        ventes_data.loc[mask, "justification"] = (
            ventes_data.loc[mask, "justification"].fillna("")
            + " ; Produit Id est absent dans la table des Magasins  !"
        )
        
        
        #nettoyage de magasin Id 
        
        #si magasin Id est absent
        mask = ventes_data["magasin_id"].isna()
        ventes_data.loc[mask, "status"] = "Rejeté"
        ventes_data.loc[mask, "justification"] = (
            ventes_data.loc[mask, "justification"].fillna("")
            + " ; Magasin Id est absent !"
        )
        #si Magasin Id n'exist pas dans lable des Magasin
        mask = (
        ventes_data["magasin_id"].notna() & ~ventes_data["magasin_id"].isin(magasins_data["magasin_id"]))
        ventes_data.loc[mask, "status"] = "Rejeté"
        ventes_data.loc[mask, "justification"] = (
            ventes_data.loc[mask, "justification"].fillna("")
            + " ; Magasin Id est absent dans la table des Magasins  !"
        )
        
        
        #nettoyage de quantité 
        
        #si quantité est absent ou non numerique
        quantite = ventes_data["quantite"].astype("string").str.strip()
        empty_mask = quantite.isna() | quantite.eq("")
        non_numeric_mask = ~empty_mask & pd.to_numeric(quantite, errors="coerce").isna()
        negatif_numeric_mask = quantite.notna() & (pd.to_numeric(quantite, errors="coerce") < 0 )
        
        ventes_data.loc[empty_mask, "status"] = "Rejeté"
        ventes_data.loc[non_numeric_mask, "status"] = "Rejeté"
        ventes_data.loc[negatif_numeric_mask, "status"] = "Rejeté"
        
        ventes_data.loc[empty_mask, "justification"] = (
            ventes_data.loc[empty_mask, "justification"].fillna("")
            + " ; quantite est absent !"
        )
        ventes_data.loc[non_numeric_mask, "justification"] = (
            ventes_data.loc[non_numeric_mask, "justification"].fillna("")
            + " ; quantite n'est pas une valeur numérique !"
        )
        ventes_data.loc[negatif_numeric_mask, "justification"] = (
            ventes_data.loc[negatif_numeric_mask, "justification"].fillna("")
            + " ; quantite est une valeur numérique négatif !"
        )
        
        
        #nettoyage de prix_unitaire  
        
        #si prix_unitaire est absent ou non numerique
        quantite = ventes_data["prix_unitaire"].astype("string").str.strip()
        empty_mask = quantite.isna() | quantite.eq("")
        non_numeric_mask = ~empty_mask & pd.to_numeric(quantite, errors="coerce").isna()
        negatif_numeric_mask = quantite.notna() & (pd.to_numeric(quantite, errors="coerce") < 0 )
        
        ventes_data.loc[empty_mask, "status"] = "Rejeté"
        ventes_data.loc[non_numeric_mask, "status"] = "Rejeté"
        ventes_data.loc[negatif_numeric_mask, "status"] = "Rejeté"
        
        ventes_data.loc[empty_mask, "justification"] = (
            ventes_data.loc[empty_mask, "justification"].fillna("")
            + " ; prix_unitaire est absent !"
        )
        ventes_data.loc[non_numeric_mask, "justification"] = (
            ventes_data.loc[non_numeric_mask, "justification"].fillna("")
            + " ; prix_unitaire n'est pas une valeur numérique !"
        )
        ventes_data.loc[negatif_numeric_mask, "justification"] = (
            ventes_data.loc[negatif_numeric_mask, "justification"].fillna("")
            + " ; prix_unitaire est une valeur numérique négatif !"
        )
    
        #creation des nouveaux fichiers
        mask_rejetes = ventes_data["status"].eq("Rejeté")
        ventes_rejetes = ventes_data.loc[mask_rejetes].copy()
        ventes_nettoyes = ventes_data.loc[~mask_rejetes].drop(columns=["status", "justification"]).copy()
        
        dossier = Path(f"{export_dir}/Cleaned Files")
        dossier.mkdir(parents=True,exist_ok=True)
        dossier1 = Path(f"{export_dir}/Rejected Files")
        dossier1.mkdir(parents=True,exist_ok=True)
        
        ventes_rejetes.to_csv(dossier1 / "ventes_rejetes.csv", index=False)
        ventes_nettoyes.to_csv(dossier / "ventes_nettoyes.csv", index=False)

        return ventes_rejetes, ventes_nettoyes


#Nettoyage de fichier produits.csv
def Nettoyer_Produits(produits_data, export_dir):
        #eliminer les espaces
        Formatter_text(produits_data)
    
        produits_data["status"] = ""
        produits_data["justification"] = ""
        
        #nettoyage de produit_id 
        
        #si id est absent
        mask = produits_data["produit_id"].isna()
        produits_data.loc[mask, "status"] = "Rejeté"
        produits_data.loc[mask, "justification"] = (
            produits_data.loc[mask, "justification"].fillna("")
            + " ; Produit Id est absente  !"
        )
        
        
        #si id est dupliqué changer id en id(1),... et le rejeté
        duplicate_mask = produits_data["produit_id"].notna() & produits_data["produit_id"].duplicated(keep="first")
        duplicate_number = produits_data.groupby("produit_id").cumcount()
        
        produits_data.loc[duplicate_mask, "produit_id"] = (
            produits_data.loc[duplicate_mask, "produit_id"].astype("string")
            + "("
            + duplicate_number.loc[duplicate_mask].astype("string")
            + ")"
        )
        produits_data.loc[duplicate_mask, "status"] = "Rejeté"
        produits_data.loc[duplicate_mask, "justification"] = (
            produits_data.loc[duplicate_mask, "justification"].fillna("")
            + " ; Produit Id est dupliqué  !"
        )
        
        #nettoyage de produit desc et  categorie
        
        #si produit desc est absent
        mask = produits_data["nom_produit"].isna()
        produits_data.loc[mask, "nom_produit"] = "Unkown"
        
        #si produit categorie est absent
        mask = produits_data["categorie"].isna()
        produits_data.loc[mask, "categorie"] = "Unkown"
        
        #creation des nouveaux fichiers
        mask_rejetes = produits_data["status"].eq("Rejeté")
        produits_rejetes = produits_data.loc[mask_rejetes].copy()
        produits_nettoyes = produits_data.loc[~mask_rejetes].drop(columns=["status", "justification"]).copy()

        dossier = Path(f"{export_dir}/Cleaned Files")
        dossier.mkdir(parents=True,exist_ok=True)
        dossier1 = Path(f"{export_dir}/Rejected Files")
        dossier1.mkdir(parents=True, exist_ok=True)
        
        produits_rejetes.to_csv(dossier1 / "produits_rejetes.csv", index=False)
        produits_nettoyes.to_csv(dossier / "produits_nettoyes.csv", index=False)
        
        return produits_rejetes, produits_nettoyes

            
#Nettoyage de fichier magasins.csv

def Nettoyer_Magasins(magasins_data, export_dir):
        #eliminer les espaces 
        Formatter_text(magasins_data) 
    
        magasins_data["status"] = ""
        magasins_data["justification"] = ""
        #nettoyage de magasin_id 
        
        #si magasin_id est absent
        mask = magasins_data["magasin_id"].isna()
        magasins_data.loc[mask, "status"] = "Rejeté"
        magasins_data.loc[mask, "justification"] = (
            magasins_data.loc[mask, "justification"].fillna("")
            + " ; Magasin Id est absente  !"
        )
        
        #si id est dupliqué changer id en id(1),... et le rejeté
        duplicate_mask = magasins_data["magasin_id"].notna() & magasins_data["magasin_id"].duplicated(keep="first")
        duplicate_number = magasins_data.groupby("magasin_id").cumcount()
        
        magasins_data.loc[duplicate_mask, "magasin_id"] = (
            magasins_data.loc[duplicate_mask, "magasin_id"].astype("string")
            + "("
            + duplicate_number.loc[duplicate_mask].astype("string")
            + ")"
        )
        magasins_data.loc[duplicate_mask, "status"] = "Rejeté"
        magasins_data.loc[duplicate_mask, "justification"] = (
            magasins_data.loc[duplicate_mask, "justification"].fillna("")
            + " ; Magasin Id est dupliqué  !"
        )
        
        #si magasin ville est absent
        mask = magasins_data["ville"].isna()
        magasins_data.loc[mask, "ville"] = "Unknown"
       

        #creation des nouveaux fichiers
        mask_rejetes = magasins_data["status"].eq("Rejeté")
        magasins_rejetes = magasins_data.loc[mask_rejetes].copy()
        magasins_nettoyes = magasins_data.loc[~mask_rejetes].drop(columns=["status", "justification"]).copy()

        dossier = Path(f"{export_dir}/Cleaned Files")
        dossier.mkdir(parents=True, exist_ok=True)
        
        dossier1 = Path(f"{export_dir}/Rejected Files")
        dossier1.mkdir(parents=True, exist_ok=True)
        
        magasins_rejetes.to_csv( dossier1 / "magasins_rejetes.csv",index=False)
        magasins_nettoyes.to_csv(dossier / "magasins_nettoyes.csv",index=False)
    
        return magasins_rejetes, magasins_nettoyes

def Fusioner(ventes_nettoyes,produits_nettoyes,magasins_nettoyes, export_dir) :
        
        #calculer et affecter le chiffre d'affaire = Q x PU
        quantite = pd.to_numeric(ventes_nettoyes["quantite"], errors="coerce")
        prix = pd.to_numeric(ventes_nettoyes["prix_unitaire"], errors="coerce")
        ventes_nettoyes["chiffre_affaire"] = quantite * prix
        #fusionner les sources : ventes , produits et magasins
        #fusionner nom et categorie du produit et ville de magasin avec ventes.csv
        noms_produits = produits_nettoyes.set_index("produit_id")["nom_produit"]
        categorie_produits = produits_nettoyes.set_index("produit_id")["categorie"]
        villes_magasin = magasins_nettoyes.set_index("magasin_id")["ville"]
        
        ventes_nettoyes["produit"] = ventes_nettoyes["produit_id"].map(noms_produits)
        ventes_nettoyes["categorie"] = ventes_nettoyes["produit_id"].map(categorie_produits)
        ventes_nettoyes["ville"] = ventes_nettoyes["magasin_id"].map(villes_magasin)

        #supprimer les colonnes non necessaires
        ventes_nettoyes = ventes_nettoyes.drop(columns=["produit_id"])
        ventes_nettoyes = ventes_nettoyes.drop(columns=["magasin_id"])


        dossier = Path(f"{export_dir}/Cleaned Files")
        dossier.mkdir(parents=True, exist_ok=True)
        
        ventes_nettoyes.to_csv(dossier / "ventes_nettoyes.csv", index=False)

        return ventes_nettoyes
#renvoyer le chiffre d'affaire total
def Chiffre_affaires_total(ventes_transformes) :
    try :
        return ventes_transformes["chiffre_affaire"].sum()
    except Exception as e:
        print(f"Erreur lors de la traitement... Try again : {e}")
        return None
#renvoyer le chiffre d'affaire Filtrer
def Chiffre_affaires_filtrer(ventes_transformes, cle) :
    try :
        if cle == "date" :
            ventes_transformes[cle] = pd.to_datetime(ventes_transformes[cle])
            ca = ventes_transformes.groupby(ventes_transformes[cle].dt.to_period("M"))["chiffre_affaire"].sum()
            return ca
        else :
            ca = ventes_transformes.groupby(cle)["chiffre_affaire"].sum()
            return ca
    except Exception as e:
        print(f"Erreur lors de la traitement... Try again : {e}")
        return None



def Analyse_chiffre_affaires(ventes_transformes, export_dir):
    chiffre_affaires_tot = Chiffre_affaires_total(ventes_transformes)
    analyses = [
        ("Chiffre d'affaires par ville",
         Chiffre_affaires_filtrer(ventes_transformes, "ville")),

        ("Chiffre d'affaires par produit",
         Chiffre_affaires_filtrer(ventes_transformes, "produit")),

        ("Chiffre d'affaires par catégorie",
         Chiffre_affaires_filtrer(ventes_transformes, "categorie")),

        ("Chiffre d'affaires par date",
         Chiffre_affaires_filtrer(ventes_transformes, "date")),
    ]

    # Création du dossier
    dossier = Path(f"{export_dir}/Rapports")
    dossier.mkdir(parents=True, exist_ok=True)
    # Chemin du fichier PDF
    chemin_pdf = dossier / "CA Details.pdf"
    # Création du PDF
    with PdfPages(chemin_pdf) as pdf:
        # Première page : résumé
        fig, ax = plt.subplots(figsize=(11, 5))
        fig.patch.set_facecolor("#F4F7FB")
        ax.axis("off")
        ax.text(
            0.5, 0.75,
            "RAPPORT D'ANALYSE DU CHIFFRE D'AFFAIRES",
            ha="center",
            fontsize=17,
            fontweight="bold",
            color="#163A5F"
        )
        ax.text(
            0.5, 0.50,
            f"Chiffre d'affaires total : {chiffre_affaires_tot:,.2f} DH",
            ha="center",
            fontsize=15,
            color="#168AAD"
        )
        ax.text(
            0.5, 0.30,
            f"Rapport généré le {datetime.now():%d/%m/%Y}",
            ha="center",
            fontsize=10,
            color="gray"
        )

        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

        # Pages suivantes : tableaux d'analyse
        for titre, resultat in analyses:
            # Création de la figure
            fig, ax = plt.subplots(figsize=(11, 6))

            fig.patch.set_facecolor("#F4F7FB")
            ax.axis("off")

            ax.set_title(
                titre.upper(),
                fontsize=15,
                fontweight="bold",
                color="#163A5F",
                pad=20
            )

            # Préparation des données
            resultat = resultat.reset_index()

            # Création du tableau
            tab = ax.table(
                cellText=resultat.values,
                colLabels=resultat.columns,
                colColours=["#163A5F"] * len(resultat.columns),
                cellLoc="center",
                loc="center",
                bbox=[0.05, 0.05, 0.9, 0.85]
            )

            # Personnalisation
            tab.auto_set_font_size(False)
            tab.set_fontsize(10)
            tab.scale(1, 1.5)

            for (ligne, colonne), cellule in tab.get_celld().items():

                cellule.set_edgecolor("#D5DEE8")

                if ligne == 0:
                    cellule.set_text_props(
                        color="white",
                        weight="bold"
                    )

                else:
                    if ligne % 2 == 0:
                        cellule.set_facecolor("#EAF2F8")
                    else:
                        cellule.set_facecolor("#FFFFFF")

            # Enregistrement dans le PDF
            pdf.savefig(fig, bbox_inches="tight")

            plt.close(fig)

    print(f"\nAnalyse du chiffre d'affire a été enregistré avec succès : {chemin_pdf}")
#Visualize data

def visualiser_chiffre_affaire(data, cle, export_dir):
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(data.index.astype(str), data.values)
    ax.set_title(f"Chiffre d'affaires par {cle}")
    ax.set_xlabel(cle)
    ax.set_ylabel("Chiffre d'affaires (DH)")
    fig.tight_layout()
    dossier = Path(f"{export_dir}/Rapports/Visualisation/CA Visualisation")
    dossier.mkdir(parents=True, exist_ok=True)
    fig.savefig(dossier / f"CA({cle}).png",dpi=150,bbox_inches="tight",pad_inches=0)
    plt.close(fig)
    print(f"\n CA({cle}) image enregistré avec succès : {dossier}")
    return fig


def visualiser_ventes(df,export_dir):
    colonnes_requises = {
        "date", "produit", "ville", "categorie", "chiffre_affaire"
    }
    manquantes = colonnes_requises - set(df.columns)
    if manquantes:
        raise ValueError(f"Colonnes manquantes : {', '.join(sorted(manquantes))}")

    data = df.copy()
    data["date"] = pd.to_datetime(data["date"], errors="coerce")
    data["chiffre_affaire"] = pd.to_numeric(
        data["chiffre_affaire"], errors="coerce"
    )
    data = data.dropna(subset=["date", "chiffre_affaire"])

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    par_mois = data.groupby(data["date"].dt.to_period("M"))["chiffre_affaire"].sum()
    axes[0, 0].plot(par_mois.index.astype(str), par_mois.values, marker="o")
    axes[0, 0].set_title("Chiffre d'affaires par mois")
    axes[0, 0].tick_params(axis="x", rotation=45)

    par_ville = data.groupby("ville")["chiffre_affaire"].sum().sort_values()
    axes[0, 1].barh(par_ville.index.astype(str), par_ville.values)
    axes[0, 1].set_title("Chiffre d'affaires par ville")

    par_categorie = data.groupby("categorie")["chiffre_affaire"].sum().sort_values()
    axes[1, 0].barh(par_categorie.index.astype(str), par_categorie.values)
    axes[1, 0].set_title("Chiffre d'affaires par catégorie")

    par_produit = data.groupby("produit")["chiffre_affaire"].sum().nlargest(10).sort_values()
    axes[1, 1].barh(par_produit.index.astype(str), par_produit.values)
    axes[1, 1].set_title("Top 10 des produits")

    for ax in axes.flat:
        ax.set_xlabel("Chiffre d'affaires (DH)")

    fig.tight_layout()
    dossier = Path(f"{export_dir}/Rapports/Visualisation")
    dossier.mkdir(parents=True, exist_ok=True)
    fig.savefig(dossier / "Ventes Visualisation.png", dpi=150, bbox_inches="tight")
    print(f"\n Ventes Visualisation Image enregistré avec succès : {dossier}")
    plt.close(fig)
    return fig
    
def Resume_des_indicateurs(ventes_transformes, export_dir):
        ca = pd.to_numeric(ventes_transformes["chiffre_affaire"], errors="coerce")
        quantite = pd.to_numeric(ventes_transformes["quantite"], errors="coerce")
        
        nb_tot_ventes = ventes_transformes["vente_id"].count()
        chiffre_affaires_tot = ca.sum()
        quantite_tot = quantite.sum()
        ca_moyenne = ca.mean()
        ca_mediane = ca.median()
        
        nb_tot_magasins = ventes_transformes["ville"].nunique()
        nb_tot_produits = ventes_transformes["produit"].nunique()
    
        produit_mode = list(ventes_transformes["produit"].mode())
        categorie_mode = list(ventes_transformes["categorie"].mode())
        date_mode = max(Chiffre_affaires_filtrer(ventes_transformes,"date"))

        # Création du DataFrame
        tableau = pd.DataFrame({
            "Indicateur": [
                "Nombre de ventes",
                "Chiffre d'affaires total (DH)",
                "CA moyen (DH)",
                "CA médian (DH)",
                "Nombre de magasins",
                "Nombre de produits",
                "Produit le plus vendu",
                "Catégorie la plus vendue",
                "Meilleur mois des ventes"
            ],
        
            "Valeur": [
                nb_tot_ventes,
                f"{chiffre_affaires_tot:,.2f}",
                f"{ca_moyenne:,.2f}",
                f"{ca_mediane:,.2f}",
                nb_tot_magasins,
                nb_tot_produits,
                produit_mode,
                categorie_mode,
                date_mode
            ]
        })
        
        # Création de la figure
        fig, ax = plt.subplots(figsize=(11, 6))
        
        fig.patch.set_facecolor("#F4F7FB")
        ax.set_facecolor("#F4F7FB")
        ax.axis("off")
        
        # Titre
        ax.set_title(
            "TABLEAU DE BORD - INDICATEURS DE PERFORMANCE",
            fontsize=15,
            fontweight="bold",
            color="#163A5F",
            pad=25
        )
        
        # Couleurs des lignes
        couleurs = [
            ["#FFFFFF", "#FFFFFF"] if i % 2 == 0
            else ["#EAF2F8", "#EAF2F8"]
            for i in range(len(tableau))
        ]
        
        # Création du tableau
        tab = ax.table(
            cellText=tableau.values,
            colLabels=tableau.columns,
            cellColours=couleurs,
            colColours=["#163A5F", "#163A5F"],
            colWidths=[0.65, 0.35],
            cellLoc="left",
            colLoc="center",
            loc="center",
            bbox=[0.05, 0.05, 0.9, 0.85]
        )
        
        # Personnalisation
        tab.auto_set_font_size(False)
        tab.set_fontsize(11)
        tab.scale(1, 1.8)
        
        # Personnalisation des cellules
        for (ligne, colonne), cellule in tab.get_celld().items():
        
            cellule.set_edgecolor("#D5DEE8")
            cellule.set_linewidth(0.5)
        
            if ligne == 0:
                cellule.set_text_props(
                    color="white",
                    weight="bold",
                    ha="center"
                )
                cellule.set_height(0.08)
        
            else:
                if colonne == 0:
                    cellule.set_text_props(
                        color="#34495E",
                        weight="bold",
                        ha="left"
                    )
                else:
                    cellule.set_text_props(
                        color="#163A5F",
                        weight="bold",
                        ha="center"
                    )
        
        # Date de génération
        fig.text(
            0.5, 0.025,
            f"Rapport généré le {datetime.now():%d/%m/%Y à %H:%M}",
            ha="center",
            fontsize=9,
            color="#7B8794"
        )
        
        # Création du dossier exports
        dossier = Path(f"{export_dir}/Rapports")
        dossier.mkdir(parents=True, exist_ok=True)
        
        # Enregistrement du tableau en PDF
        fig.savefig(
            dossier / "Rapport KPI.pdf",
            format="pdf",
            bbox_inches="tight",
            dpi=300,
            facecolor=fig.get_facecolor()
        )
        plt.close(fig)

        # Display success message
        print(f"\nRapport successfully exported to: {dossier / 'Rapport KPI.pdf'}")                
#function that run the pipline 
def Pipline(path_ventes, path_produits,path_magasins, export_dir):
     #Initialiser les fichiers
    try : 
        ventes_data = Lire_fichier(path_ventes)
        produits_data = Lire_fichier(path_produits)
        magasins_data = Lire_fichier(path_magasins)
        
        
    
        
        #traitement de fichier de ventes nettouyer
        ventes_nettoyes = Nettoyer_Ventes(ventes_data,produits_data,magasins_data,export_dir)[1]
        produits_nettoyes = Nettoyer_Produits(produits_data,export_dir)[1]
        magasins_nettoyes = Nettoyer_Magasins(magasins_data,export_dir)[1]
    
        #appel du fonction qui fusionne
        ventes_transformes = Fusioner(ventes_nettoyes,produits_nettoyes,magasins_nettoyes, export_dir)
    
        #Analyse des données
        
        Analyse_chiffre_affaires(ventes_transformes, export_dir)
    
        ca_date = Chiffre_affaires_filtrer(ventes_transformes,"date")
        ca_categorie = Chiffre_affaires_filtrer(ventes_transformes,"categorie")
        ca_ville = Chiffre_affaires_filtrer(ventes_transformes,"ville")
        ca_produit = Chiffre_affaires_filtrer(ventes_transformes,"produit")
    
        #Visualize Data
    
        
        visualiser_chiffre_affaire(ca_date,"date", export_dir)   
        visualiser_chiffre_affaire(ca_categorie,"categorie", export_dir)
        visualiser_chiffre_affaire(ca_ville,"ville", export_dir)
        visualiser_chiffre_affaire(ca_produit,"produit", export_dir)
        
        visualiser_ventes(ventes_transformes, export_dir)
    
        #statistiques descriptives     
        Resume_des_indicateurs(ventes_transformes, export_dir)
    except Exception as e :
        print(f"Erreur lors du traitement ...\n {e}")

#Verifier les shema des fichier 
def Verifier_shema(ventes_data, produits_data, magasins_data):
    
        schemas = {
            "ventes": [
                "vente_id", "date", "produit_id",
                "magasin_id", "quantite", "prix_unitaire"
            ],
    
            "produits": [
                "produit_id", "nom_produit", "categorie"
            ],
    
            "magasins": [
                "magasin_id", "ville"
            ]
        }
    
        dataframes = {
            "ventes": ventes_data,
            "produits": produits_data,
            "magasins": magasins_data
        }
    
        erreurs = []
    
        for nom, df in dataframes.items():
    
            colonnes_attendues = schemas[nom]
            colonnes_actuelles = df.columns.tolist()
    
            # Vérification des colonnes manquantes
            manquantes = [
                col for col in colonnes_attendues
                if col not in colonnes_actuelles
            ]
    
            # Vérification des colonnes supplémentaires
            supplementaires = [
                col for col in colonnes_actuelles
                if col not in colonnes_attendues
            ]
    
            if manquantes or supplementaires:
    
                erreurs.append(f"\nDataFrame : {nom}")
    
                if manquantes:
                    erreurs.append(f"Colonnes manquantes : {manquantes}")
    
                if supplementaires:
                    erreurs.append(f"Colonnes supplémentaires : {supplementaires}")
    
        if erreurs:
            print("ERREUR : Schéma des fichiers invalide.")
    
            for erreur in erreurs:
                print(erreur)
    
            return False
    
        print("Les trois DataFrames possèdent un schéma valide.")
        return True
#Main programme

if __name__ == "__main__":
    while True :
        try : 
            # Request file paths
            ventes_p = input("Entrer le fichier des ventes : ").strip()
            produits_p = input("Entrer le fichier des produits : ").strip()
            magasins_p = input("Entrer le fichier des magasins : ").strip()
            # Verify schemas
            ventes = str(Path(ventes_p))
            produits = str(Path(produits_p))
            magasins = str(Path(magasins_p))
            if Verifier_shema(
                Lire_fichier(ventes),
                Lire_fichier(produits),
                Lire_fichier(magasins)
                             ):
                print("\nLancement du pipeline...")
                
                # Create a unique folder for each execution
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                export_dir = Path("Exports") / f"pipeline_{timestamp}"
                
                Pipline(ventes_p, produits_p, magasins_p, export_dir)
                print("\nPipeline exécuté avec succès !")
            else:
                print("\nPipeline interrompu.")
            #choix de continue ou arreter
            choix = input("\nVoulez-vous continuer ? (y/n) : ").strip().lower()
            if choix == "n":
                print("Programme terminé.")
                break
            elif choix == "y":
                continue
            else:
                print("Choix invalide. Veuillez saisir y ou n.")
        except Exception as e : 
            print(f"\n Error occurend lors du traitement {e}")
    


       


Entrer le fichier des ventes :  Data Files2/ventes.csv
Entrer le fichier des produits :  Data Files2/produits.csv
Entrer le fichier des magasins :  Data Files2/magasins.csv


Les trois DataFrames possèdent un schéma valide.

Lancement du pipeline...

Analyse du chiffre d'affire a été enregistré avec succès : Exports\pipeline_20261105_102106\Rapports\CA Details.pdf

 CA(date) image enregistré avec succès : Exports\pipeline_20261105_102106\Rapports\Visualisation\CA Visualisation

 CA(categorie) image enregistré avec succès : Exports\pipeline_20261105_102106\Rapports\Visualisation\CA Visualisation

 CA(ville) image enregistré avec succès : Exports\pipeline_20261105_102106\Rapports\Visualisation\CA Visualisation

 CA(produit) image enregistré avec succès : Exports\pipeline_20261105_102106\Rapports\Visualisation\CA Visualisation

 Ventes Visualisation Image enregistré avec succès : Exports\pipeline_20261105_102106\Rapports\Visualisation

Rapport successfully exported to: Exports\pipeline_20261105_102106\Rapports\Rapport KPI.pdf

Pipeline exécuté avec succès !



Voulez-vous continuer ? (y/n) :  n


Programme terminé.
